# 8. Tuneo, promedio de semillas y CatBoost

Última iteración del episodio. El mejor envío es el blend 50/50 de `wide_te` (notebook 7) con `newton`
(notebook 6): **OOF 0,945997**. El pelotón del leaderboard está en 0,9466, o sea a ~0,0003-0,0004.

**Lo que ya se descartó por medición, para no repetirlo:**

| Vía | Resultado |
|---|---|
| Stacking de los 7 OOF guardados | +0,000019 sobre el blend 50/50, con CV anidada. Agregar modelos empeora. |
| Más modelos de la misma familia | Todos los pares del episodio correlacionan ≥0,977; el menos correlacionado (logit, 0,9775) rinde 0,0073 menos. |
| Dataset original como encoding | 8.571 de 9.093 valores de ingreso aparecen una vez. AUC 0,516. |
| Artefactos del generador | +0,0021 sobre el modelo base, pero **−0,000019** sobre el encoding ancho: redundantes. |
| Dígitos del ingreso, `commute` por valor, `inc×env` | Ratio de varianza residual ≤1, o sea nada. |

**Las tres cosas que quedan, y lo que se sabe de cada una:**

1. **Promediar semillas** — *medido*. En el fold 1 con tres semillas (variando la del `TargetEncoder` y
   la de LightGBM): AUC individual 0,944887 / 0,944883 / 0,944877, o sea **sd de 0,000005** entre
   semillas. Pero el promedio de las tres da **0,944985**, o sea **+0,000098** sobre la mejor
   individual. La correlación de rangos entre semillas es 0,99817: rinden idéntico pero discrepan en el
   ordenamiento, que es exactamente la condición para que promediar ayude. Es reducción de varianza:
   no hay hipótesis que pueda fallar ni sesgo de selección.
2. **Tunear el LightGBM** — *no medido*. `wide_te` corre con parámetros puestos a mano y nunca
   buscados. Es la misma situación del notebook 3 antes de Optuna, que rindió +0,0007; acá esperaría
   menos porque las features son mucho más fuertes.
3. **CatBoost** — *no medido*. Es el único candidato con chance de romper el muro de 0,99: sus
   *ordered target statistics* llegan a la misma señal del ingreso por otro camino. Se le dan las
   claves multi-escala **como categóricas**, no nuestras columnas TE ya calculadas — darle el encoding
   hecho lo convertiría en una copia correlacionada.

> **Advertencia de calibre.** Sólo el punto 1 tiene un número medido detrás, y es +0,0001. Los otros dos
> son estimaciones por analogía, y este episodio ya desmintió dos veces mis magnitudes: subestimé los
> artefactos del generador 4-10× y sobreestimé el efecto de `ALPHA` 8×.

**Métrica:** ROC AUC · **Dataset:** Playground Series S6E9

In [ ]:
import time, json
from pathlib import Path
import numpy as np
import pandas as pd
import lightgbm as lgb
import optuna
from optuna.samplers import TPESampler
from catboost import CatBoostClassifier, Pool
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import TargetEncoder
from sklearn.metrics import roc_auc_score
from scipy.stats import rankdata, ttest_rel
import warnings
warnings.filterwarnings("ignore")
optuna.logging.set_verbosity(optuna.logging.WARNING)

SEED, N_FOLDS = 42, 5
TARGET = "Will_Buy_EV"
ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / "data/train.csv").exists()), None)
if ROOT is None:
    raise FileNotFoundError("Abrir desde notebook/ o desde la raiz del episodio.")
OUT = ROOT / "models" / "8_final"
OUT.mkdir(parents=True, exist_ok=True)
print("lightgbm", lgb.__version__, "| optuna", optuna.__version__, "| salida:", OUT)

## 1. Features

Se reproduce la receta de `wide_te`, que fue el mejor individual del notebook 7 (OOF 0,945838). No se
incluyen los artefactos del generador: medían **−0,000019** encima del encoding ancho, con 1 fold
ganado de 5.

Para CatBoost las mismas claves entran **como categóricas de texto**, para que calcule sus propias
estadísticas de target en vez de recibir las nuestras.

In [ ]:
train = pd.read_csv(ROOT / "data/train.csv")
test  = pd.read_csv(ROOT / "data/test.csv")
sample = pd.read_csv(ROOT / "data/sample_submission.csv")
assert test["id"].equals(sample["id"])
y = train[TARGET].eq("Yes").to_numpy(np.int8)

CAT = ["Gender","City_Type","Current_Car_Type",
       "Home_Charging_Possible","Subsidy_Available","Range_Anxiety_Level"]
NUM = ["Age","Annual_Income_USD","Daily_Commute_km","Number_of_Cars_Owned",
       "Charging_Stations_Near_Home","Charging_Stations_Near_Work",
       "Environmental_Concern_Level"]

def preparar(df):
    d = df.copy()
    inc, com = d["Annual_Income_USD"], d["Daily_Commute_km"]
    d["inc_100"]  = np.floor(inc / 100) * 100
    d["inc_1000"] = np.floor(inc / 1000) * 1000
    d["com_1"]    = np.floor(com)
    # versiones de texto para CatBoost
    d["k_inc"]     = inc.astype(np.int64).astype(str)
    d["k_inc100"]  = d["inc_100"].astype(np.int64).astype(str)
    d["k_inc1000"] = d["inc_1000"].astype(np.int64).astype(str)
    d["k_com"]     = np.round(com * 10).astype(np.int64).astype(str)
    d["k_com1"]    = d["com_1"].astype(np.int64).astype(str)
    return d

TR, TE_ = preparar(train), preparar(test)
for c in CAT:
    dt = pd.CategoricalDtype(sorted(set(TR[c]) | set(TE_[c])))
    TR[c] = TR[c].astype(dt); TE_[c] = TE_[c].astype(dt)

CLAVES  = ["Annual_Income_USD","inc_100","inc_1000","Daily_Commute_km","com_1"]
TE_COLS = CLAVES + CAT + ["Age","Environmental_Concern_Level",
                          "Charging_Stations_Near_Home","Charging_Stations_Near_Work"]
SMOOTHS = [("auto","auto"), ("10", 10.0), ("100", 100.0)]
TE_ALL  = [f"te{t}_{c}" for t,_ in SMOOTHS for c in TE_COLS]
FEATS_LGB = NUM + CAT + TE_ALL

KEYS_CB   = ["k_inc","k_inc100","k_inc1000","k_com","k_com1"]
FEATS_CB  = NUM + CAT + KEYS_CB
CAT_IDX   = [FEATS_CB.index(c) for c in CAT + KEYS_CB]
for c in CAT: TR[c] = TR[c].astype(str); TE_[c] = TE_[c].astype(str)
print(f"LightGBM: {len(FEATS_LGB)} features | CatBoost: {len(FEATS_CB)}, "
      f"{len(CAT_IDX)} categoricas")
print("niveles de las claves de CatBoost:", {c: TR[c].nunique() for c in KEYS_CB})

In [ ]:
def encodear(Xtr, ytr, otros, te_seed):
    """Aplica el triple TargetEncoder. fit_transform hace cross-fitting interno,
    asi que la fila de entrenamiento no ve su propia etiqueta."""
    Xtr = Xtr.copy(); otros = [o.copy() for o in otros]
    for tag, sm in SMOOTHS:
        enc = TargetEncoder(smooth=sm, cv=5, shuffle=True, random_state=te_seed)
        a = enc.fit_transform(Xtr[TE_COLS].astype(str), ytr)
        trans = [enc.transform(o[TE_COLS].astype(str)) for o in otros]
        for i, col in enumerate(TE_COLS):
            Xtr[f"te{tag}_{col}"] = a[:, i]
            for o, t in zip(otros, trans):
                o[f"te{tag}_{col}"] = t[:, i]
    return Xtr, otros

## 2. Búsqueda de hiperparámetros

Mismo esquema del notebook 4, que funcionó: submuestra de 250k, 3 folds y `learning_rate` alto para la
búsqueda; el modelo final vuelve al régimen completo.

Con un ahorro clave: **el encoding no depende de los hiperparámetros**, así que se calcula una sola vez
para los folds de búsqueda y se reutiliza en todos los trials. Sin eso, cada trial pagaría ~25 s de
`TargetEncoder` al aire.

In [ ]:
N_TRIALS, N_SUB, LR_BUSQUEDA = 40, 250_000, 0.1
rng = np.random.default_rng(SEED)
sub = np.sort(rng.choice(len(TR), N_SUB, replace=False))
Xs, ys = TR.iloc[sub].reset_index(drop=True), y[sub]

t0 = time.perf_counter()
SEARCH = []
for itr, iva in StratifiedKFold(3, shuffle=True, random_state=SEED).split(Xs, ys):
    a, (b,) = encodear(Xs.iloc[itr], ys[itr], [Xs.iloc[iva]], SEED)
    SEARCH.append((a[FEATS_LGB], ys[itr], b[FEATS_LGB], ys[iva]))
print(f"encoding de busqueda precalculado en {(time.perf_counter()-t0)/60:.1f} min "
      f"(se reusa en los {N_TRIALS} trials)")

In [ ]:
BASE_LGB = dict(n_estimators=3000, subsample_freq=1, n_jobs=4, verbose=-1)

def objetivo(trial):
    p = dict(
        learning_rate=LR_BUSQUEDA,
        num_leaves=trial.suggest_int("num_leaves", 16, 256, log=True),
        min_child_samples=trial.suggest_int("min_child_samples", 10, 200, log=True),
        subsample=trial.suggest_float("subsample", 0.6, 1.0),
        colsample_bytree=trial.suggest_float("colsample_bytree", 0.3, 1.0),
        reg_lambda=trial.suggest_float("reg_lambda", 1e-2, 50.0, log=True),
        reg_alpha=trial.suggest_float("reg_alpha", 1e-3, 10.0, log=True),
        min_split_gain=trial.suggest_float("min_split_gain", 0.0, 1.0),
        max_bin=trial.suggest_categorical("max_bin", [127, 255, 511]),
    )
    aucs = []
    for Xa, ya, Xb, yb in SEARCH:
        m = lgb.LGBMClassifier(**BASE_LGB, **p, random_state=SEED)
        m.fit(Xa, ya, eval_set=[(Xb, yb)], eval_metric="auc",
              callbacks=[lgb.early_stopping(50, verbose=False)])
        aucs.append(roc_auc_score(yb, m.predict_proba(Xb)[:, 1]))
    return float(np.mean(aucs))

def cb(est, tr_):
    marca = "*" if est.best_trial.number == tr_.number else " "
    print(f"  {marca} trial {tr_.number:>2}: {tr_.value:.6f} | mejor {est.best_value:.6f}", flush=True)

t0 = time.perf_counter()
study = optuna.create_study(direction="maximize", sampler=TPESampler(seed=SEED))
study.optimize(objetivo, n_trials=N_TRIALS, callbacks=[cb])
print(f"\nbusqueda: {(time.perf_counter()-t0)/60:.1f} min")
print(f"mejor en la submuestra: {study.best_value:.6f}")
trials = study.trials_dataframe()
print(f"trial 0 (primera al azar): {trials.iloc[0]['value']:.6f}  "
      f"-> la busqueda gano {study.best_value - trials.iloc[0]['value']:+.6f}")
for k, v in study.best_params.items():
    print(f"  {k:20s} = {v}")

## 3. Modelo final: LightGBM tuneado, promediado sobre semillas

Régimen completo: train entero, 5 folds, `learning_rate=0,05`, mismos folds que todo el episodio.

Se promedian **seis modelos**: tres semillas del `TargetEncoder` × dos de LightGBM. La semilla del
encoder es la que más diversidad aporta (fue la que varió en la medición del +0,000098), y para cada
encoding entrenar dos LightGBM cuesta poco porque el encoding ya está hecho.

Se guarda también el modelo de **una sola semilla** para poder medir, contra los mismos folds, cuánto
aportó promediar y cuánto el tuneo.

In [ ]:
PARAMS_FINAL = dict(**BASE_LGB, **study.best_params, learning_rate=0.05)
TE_SEEDS, LGB_SEEDS = [42, 202, 1337], [42, 777]
skf = StratifiedKFold(N_FOLDS, shuffle=True, random_state=SEED)

oof  = {"lgb_tuned_1seed": np.zeros(len(TR)), "lgb_tuned_avg": np.zeros(len(TR))}
pred = {"lgb_tuned_1seed": np.zeros(len(TE_)), "lgb_tuned_avg": np.zeros(len(TE_))}
fold_ids = np.full(len(TR), -1, dtype=np.int8)
t0 = time.perf_counter()

for fold, (itr, iva) in enumerate(skf.split(TR, y), 1):
    fold_ids[iva] = fold
    rv, rt = [], []
    for ts in TE_SEEDS:
        Xa, (Xb, Xc) = encodear(TR.iloc[itr], y[itr], [TR.iloc[iva], TE_], ts)
        for ls in LGB_SEEDS:
            m = lgb.LGBMClassifier(**PARAMS_FINAL, random_state=ls)
            m.fit(Xa[FEATS_LGB], y[itr], eval_set=[(Xb[FEATS_LGB], y[iva])],
                  eval_metric="auc", callbacks=[lgb.early_stopping(100, verbose=False)])
            pv = m.predict_proba(Xb[FEATS_LGB])[:, 1]
            pt = m.predict_proba(Xc[FEATS_LGB])[:, 1]
            rv.append(rankdata(pv) / len(pv)); rt.append(rankdata(pt) / len(pt))
            if ts == TE_SEEDS[0] and ls == LGB_SEEDS[0]:
                oof["lgb_tuned_1seed"][iva] = pv
                pred["lgb_tuned_1seed"] += pt / N_FOLDS
    oof["lgb_tuned_avg"][iva] = np.mean(rv, axis=0)
    pred["lgb_tuned_avg"] += np.mean(rt, axis=0) / N_FOLDS
    print(f"Fold {fold}: 1 semilla={roc_auc_score(y[iva], oof['lgb_tuned_1seed'][iva]):.6f}"
          f" | {len(rv)} promediadas={roc_auc_score(y[iva], oof['lgb_tuned_avg'][iva]):.6f}", flush=True)

print(f"\nLightGBM final: {(time.perf_counter()-t0)/60:.1f} min")

## 4. CatBoost

Único modelo del episodio que no recibe el encoding hecho: las claves de ingreso y distancia entran
como categóricas y CatBoost calcula sus propias *ordered target statistics*. Es la apuesta a la
diversidad, no al score individual.

> En una prueba de un fold resultó **mucho más lento** que LightGBM con estas cardinalidades (el
> ingreso exacto tiene ~13.000 niveles). Los parámetros de abajo están elegidos para que la corrida
> cierre en tiempo razonable, no para maximizar: si CatBoost queda lejos, el diagnóstico útil sigue
> siendo su correlación de rangos.

In [ ]:
PARAMS_CB = dict(iterations=1200, learning_rate=0.08, depth=6, l2_leaf_reg=6.0,
                 eval_metric="AUC", random_seed=SEED, od_type="Iter", od_wait=80,
                 verbose=False, thread_count=4, max_ctr_complexity=1)
oof["catboost"] = np.zeros(len(TR)); pred["catboost"] = np.zeros(len(TE_))
t0 = time.perf_counter()
for fold, (itr, iva) in enumerate(skf.split(TR, y), 1):
    m = CatBoostClassifier(**PARAMS_CB)
    m.fit(Pool(TR.iloc[itr][FEATS_CB], y[itr], cat_features=CAT_IDX),
          eval_set=Pool(TR.iloc[iva][FEATS_CB], y[iva], cat_features=CAT_IDX),
          use_best_model=True)
    pv = m.predict_proba(TR.iloc[iva][FEATS_CB])[:, 1]
    oof["catboost"][iva] = pv
    pred["catboost"] += m.predict_proba(TE_[FEATS_CB])[:, 1] / N_FOLDS
    print(f"Fold {fold}: AUC={roc_auc_score(y[iva], pv):.6f} iter={m.get_best_iteration()}", flush=True)
print(f"\nCatBoost: {(time.perf_counter()-t0)/60:.1f} min")

## 5. Comparación y mezclas

Todas las mezclas usan **pesos fijados de antemano**, sobre rangos. Se reporta también el mejor peso
buscado, con la advertencia de que está sesgado: se elige sobre el mismo OOF con el que se mide.

In [ ]:
z6 = np.load(ROOT/"models/6_hierarchical/predictions.npz")
z7 = np.load(ROOT/"models/7_wide/predictions.npz")
oof["nb6_newton"]  = z6["newton_oof"];  pred["nb6_newton"]  = z6["newton_test"]
oof["nb7_wide_te"] = z7["wide_te_oof"]; pred["nb7_wide_te"] = z7["wide_te_test"]
R = lambda v: rankdata(v) / len(v)
r = {k: R(v) for k, v in oof.items()}
REF = roc_auc_score(y, 0.5*r["nb7_wide_te"] + 0.5*r["nb6_newton"])

filas = []
for k, v in oof.items():
    a = roc_auc_score(y, v)
    pf = [roc_auc_score(y[fold_ids==f], v[fold_ids==f]) for f in range(1, N_FOLDS+1)]
    filas.append({"modelo": k, "auc_oof": a, "media_fold": np.mean(pf), "sd_fold": np.std(pf, ddof=1)})
res = pd.DataFrame(filas).set_index("modelo").sort_values("auc_oof", ascending=False)
print(res.round(6).to_string())
print(f"\nreferencia (mejor envio actual, blend 50/50 nb7+nb6): {REF:.6f}")

print("\nEfectos aislados (pareado por fold):")
def pareado(a, b, que):
    pa = [roc_auc_score(y[fold_ids==f], oof[a][fold_ids==f]) for f in range(1, N_FOLDS+1)]
    pb = [roc_auc_score(y[fold_ids==f], oof[b][fold_ids==f]) for f in range(1, N_FOLDS+1)]
    d = np.array(pa) - np.array(pb)
    t = ttest_rel(pa, pb)
    print(f"  {que:36s} {d.mean():+.6f} | folds {int((d>0).sum())}/{N_FOLDS}"
          f" | sd {d.std(ddof=1):.6f} | t={t.statistic:6.2f}")
pareado("lgb_tuned_1seed", "nb7_wide_te", "tuneo (misma semilla, sin promediar)")
pareado("lgb_tuned_avg", "lgb_tuned_1seed", "promediar 6 semillas")
pareado("catboost", "nb7_wide_te", "catboost vs wide_te")

print("\nCorrelacion de rangos (el muro del episodio esta en 0,99):")
for k in ["lgb_tuned_avg", "catboost"]:
    print(f"  {k:16s} vs nb6_newton {np.corrcoef(r[k], r['nb6_newton'])[0,1]:.5f}"
          f" | vs nb7_wide_te {np.corrcoef(r[k], r['nb7_wide_te'])[0,1]:.5f}")

print("\nMezclas con pesos prefijados:")
MEZCLAS = {
    "avg+newton_50":        {"lgb_tuned_avg": .5, "nb6_newton": .5},
    "avg+cat_50":           {"lgb_tuned_avg": .5, "catboost": .5},
    "avg+newton+cat_33":    {"lgb_tuned_avg": 1/3, "nb6_newton": 1/3, "catboost": 1/3},
    "avg50+newton25+cat25": {"lgb_tuned_avg": .5, "nb6_newton": .25, "catboost": .25},
}
mezcla_oof, mezcla_test = {}, {}
for nombre, w in MEZCLAS.items():
    mezcla_oof[nombre] = sum(p * r[k] for k, p in w.items())
    mezcla_test[nombre] = sum(p * R(pred[k]) for k, p in w.items())
    a = roc_auc_score(y, mezcla_oof[nombre])
    print(f"  {nombre:24s} {a:.6f}  ({a-REF:+.6f} vs el envio actual)")

## 6. Artefactos y submissions

In [ ]:
payload = {"train_ids": train["id"].to_numpy(), "test_ids": test["id"].to_numpy(),
           "fold_ids": fold_ids, "y": y}
for k in ["lgb_tuned_1seed","lgb_tuned_avg","catboost"]:
    payload[k+"_oof"] = oof[k]; payload[k+"_test"] = pred[k]
for k, v in mezcla_oof.items():
    payload["mix_"+k+"_oof"] = v; payload["mix_"+k+"_test"] = mezcla_test[k]
np.savez_compressed(OUT/"predictions.npz", **payload)
np.save(ROOT/"models/8_lgb_avg_oof.npy",  oof["lgb_tuned_avg"])
np.save(ROOT/"models/8_lgb_avg_test.npy", pred["lgb_tuned_avg"])
res.to_csv(OUT/"metrics.csv")
(OUT/"config.json").write_text(json.dumps(
    {"seed": SEED, "folds": N_FOLDS, "trials": N_TRIALS, "n_sub": N_SUB,
     "best_params": study.best_params, "params_cb": {k: v for k, v in PARAMS_CB.items()
                                                    if k != "eval_metric"},
     "te_seeds": TE_SEEDS, "lgb_seeds": LGB_SEEDS, "mezclas": {k: {kk: round(vv,4)
        for kk, vv in w.items()} for k, w in MEZCLAS.items()}}, indent=2), encoding="utf-8")

dest = ROOT/"submissions"; dest.mkdir(exist_ok=True)
for k in ["lgb_tuned_avg","catboost"]:
    pd.DataFrame({"id": test["id"], TARGET: pred[k]}).to_csv(
        dest/f"8_{k}_submission.csv", index=False)
for k, v in mezcla_test.items():
    sub = pd.DataFrame({"id": test["id"], TARGET: v})
    assert sub["id"].equals(sample["id"]) and sub[TARGET].between(0,1).all()
    sub.to_csv(dest/f"8_mix_{k}_submission.csv", index=False)
print("CSVs en", dest)
print("Nada se envia automaticamente. Comparar contra 0,945997 del envio actual.")

## 7. Resultados

_Pendiente: se completa después de ejecutar._